In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

In [85]:
# ─────────────────────────────────────────────
# 1. CHARGEMENT DES DONNÉES
# ─────────────────────────────────────────────
df = pd.read_csv("../data/processed/clean_data_final.csv")

df.columns = df.columns.str.strip()

print("="*60)
print("         HOUSE PRICE MODEL - PRO VERSION")
print("="*60)
print("Shape:", df.shape)




         HOUSE PRICE MODEL - PRO VERSION
Shape: (2000, 14)


In [86]:
df["PoolArea"] = df["PoolArea"].fillna(0)

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

In [87]:
# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING (IMPORTANT ONLY)
# ─────────────────────────────────────────────
df["Age"] = 2025 - df["YearBuilt"]
df["RoomsTotal"] = df["Bedrooms"] + df["Bathrooms"]

In [88]:
# ─────────────────────────────────────────────
# 4. OUTLIERS REMOVAL (VERY IMPORTANT)
# ─────────────────────────────────────────────
q1 = df["Price"].quantile(0.01)
q99 = df["Price"].quantile(0.99)
df = df[(df["Price"] >= q1) & (df["Price"] <= q99)]

In [89]:
# ─────────────────────────────────────────────
# 5. FEATURES SELECTION (CLEAN VERSION)
# ─────────────────────────────────────────────
features = [
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "Location",
    "Age",
    "RoomsTotal"
]

X = df[features]
y = np.log1p(df["Price"])

cat_features = ["Location"]
num_features = ["Area", "Bedrooms", "Bathrooms", "Floors", "Age", "RoomsTotal"]



In [90]:
# ─────────────────────────────────────────────
# 6. PREPROCESSING
# ─────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", "passthrough", num_features)
])

In [91]:
# ─────────────────────────────────────────────
# 7. MODEL (XGBOOST - BEST FOR THIS TASK)
# ─────────────────────────────────────────────
model = Pipeline([
    ("prep", preprocessor),
    ("reg", XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

In [94]:
# ─────────────────────────────────────────────
# 8. CROSS VALIDATION
# ─────────────────────────────────────────────
print("\n[1/3] Cross-validation...")

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="r2")

print("CV R2:", scores.round(3))
print("Mean :", scores.mean().round(4))


[1/3] Cross-validation...
CV R2: [-0.327 -0.282 -0.362 -0.349 -0.292]
Mean : -0.3224


In [93]:
# ─────────────────────────────────────────────
# 9. TRAIN / TEST
# ─────────────────────────────────────────────
print("\n[2/3] Training...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

pred_log = model.predict(X_test)

y_pred = np.expm1(pred_log)
y_real = np.expm1(y_test)

r2 = r2_score(y_real, y_pred)
mae = mean_absolute_error(y_real, y_pred)

print("\n" + "="*60)
print("R2 Score:", round(r2, 4))
print("MAE     :", round(mae, 2))
print("="*60)



[2/3] Training...

R2 Score: -0.4741
MAE     : 267010.35


In [92]:
# ─────────────────────────────────────────────
# 10. RESULT MESSAGE
# ─────────────────────────────────────────────
if r2 >= 0.70:
    print("✅ OBJECTIF ATTEINT (+0.70)")
else:
    print("⚠ Still low → dataset needs stronger features or better location encoding")

print("\nDone ✅")

⚠ Still low → dataset needs stronger features or better location encoding

Done ✅
